# 13c Future Impact Ranking

Leakage-free future-impact ranking workflow. Existing raw inputs are read but not modified.

In [1]:

from pathlib import Path
import hashlib, re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

pd.set_option('display.max_columns', 180)
pd.set_option('display.max_rows', 160)
plt.style.use('seaborn-v0_8-whitegrid')
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
REPORT_DIR = PROJECT_ROOT / 'outputs' / 'reports'
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
for d in [TABLE_DIR, FIGURE_DIR, REPORT_DIR, PROC_DIR]: d.mkdir(parents=True, exist_ok=True)
RANDOM_SEED=42
BOOTSTRAPS=2000
PERIODS=[('1967-1985',1967,1985),('1986-2004',1986,2004),('2005-2023',2005,2023)]
TRANSITIONS=[('historical_replication','1967-1985','1986-2004'),('primary_temporal_test','1986-2004','2005-2023')]

def sha256_file(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

def norm_text(x):
    if pd.isna(x): return ''
    s=str(x).strip().lower()
    s=s.replace('&',' and ')
    s=re.sub(r'[^a-z0-9]+',' ',s)
    return re.sub(r'\s+',' ',s).strip()

def split_list_cell(x):
    if pd.isna(x): return []
    s=str(x).strip()
    if not s or s.lower() in {'nan','none'}: return []
    return [p.strip() for p in s.split(',') if p.strip() and p.strip().lower() not in {'nan'}]

def numeric_series(s):
    return pd.to_numeric(s, errors='coerce').astype('float64')

def parse_code(x):
    if pd.isna(x): return ''
    s=str(x).strip()
    if not s or s.lower() in {'nan','none'}: return ''
    if re.fullmatch(r'\d+\.0', s): s=s[:-2]
    return s

def safe_spearman(x,y):
    dat=pd.concat([pd.Series(x),pd.Series(y)],axis=1).replace([np.inf,-np.inf],np.nan).dropna()
    if len(dat)<4 or dat.iloc[:,0].nunique()<2 or dat.iloc[:,1].nunique()<2: return np.nan
    return float(stats.spearmanr(dat.iloc[:,0], dat.iloc[:,1]).statistic)

def safe_kendall(x,y):
    dat=pd.concat([pd.Series(x),pd.Series(y)],axis=1).replace([np.inf,-np.inf],np.nan).dropna()
    if len(dat)<4 or dat.iloc[:,0].nunique()<2 or dat.iloc[:,1].nunique()<2: return np.nan
    return float(stats.kendalltau(dat.iloc[:,0], dat.iloc[:,1]).statistic)

RAW_HASHES_BEFORE={p.name: sha256_file(p) for p in sorted(RAW_DIR.glob('*.csv'))}
panel=pd.read_csv(TABLE_DIR/'predictive_ranking_district_period_panel.csv')
state_comp=pd.read_csv(TABLE_DIR/'predictive_ranking_state_period_completeness.csv')
# Build transition rows.
def wide_transition(name,pred_period,out_period):
    pred=panel[panel['period']==pred_period].copy(); out=panel[panel['period']==out_period][['baseline_district_id','fatality_count','fatal_event_count','injury_count']].copy()
    out=out.rename(columns={'fatality_count':'future_fatalities','fatal_event_count':'future_fatal_event_count','injury_count':'future_injuries'})
    d=pred.merge(out,on='baseline_district_id',how='inner')
    d['transition']=name; d['predictor_period']=pred_period; d['outcome_period']=out_period
    d['future_top10_fatalities']=False
    return d
dev=wide_transition('development','1967-1985','1986-2004')
test=wide_transition('final_test','1986-2004','2005-2023')
assert len(dev)==len(test)
N=len(test); TOPK=int(np.ceil(0.10*N))
# Deterministic actual top-10 tie handling: rank by fatalities desc, then state/district ascending. Ties at cutoff are not expanded.
def mark_top(df, score_col, out_col='future_fatalities', k=TOPK, ascending=False):
    tie_cols = [c for c in ['State_Name','district_name'] if c in df.columns]
    ordered = df.sort_values([score_col] + tie_cols, ascending=[ascending] + [True] * len(tie_cols))
    return set(ordered.head(k).index)
def add_actual_top(df):
    idx=mark_top(df,'future_fatalities',k=TOPK,ascending=False); out=df.copy(); out['actual_top10']=out.index.isin(idx); out['actual_rank']=out.sort_values(['future_fatalities','State_Name','district_name'],ascending=[False,True,True]).reset_index().reset_index().set_index('index')['level_0'].add(1).reindex(out.index).astype(int); return out
dev=add_actual_top(dev); test=add_actual_top(test)
# Outcome distribution.
outdist=[]
for name,d in [('development',dev),('final_test',test)]:
    outdist.append({'transition':name,'districts':len(d),'total_future_fatalities':float(d['future_fatalities'].sum()),'positive_future_fatality_districts':int((d['future_fatalities']>0).sum()),'zero_future_fatality_districts':int((d['future_fatalities']==0).sum()),'zero_future_fatality_percent':float((d['future_fatalities']==0).mean()*100),'max_future_fatalities':float(d['future_fatalities'].max()),'top10_count':TOPK})
pd.DataFrame(outdist).to_csv(TABLE_DIR/'predictive_ranking_outcome_distribution.csv',index=False)

# Feature formulas.
base_feature_cols=['event_count','mean_flood_duration','total_flood_duration','fatality_count','injury_count','fatal_event_count','distinct_flood_years','years_since_recent_flood','average_annual_event_count','average_annual_fatalities','average_annual_injuries']
for d in [dev,test]:
    d['log_event_count']=np.log1p(d['event_count'])
    d['log_mean_duration']=np.log1p(d['mean_flood_duration'])
    d['log_total_duration']=np.log1p(d['total_flood_duration'])
    d['log_past_fatalities']=np.log1p(d['fatality_count'])
    d['log_past_injuries']=np.log1p(d['injury_count'])
    d['log_fatal_event_count']=np.log1p(d['fatal_event_count'])
    d['log_distinct_flood_years']=np.log1p(d['distinct_flood_years'])
    d['recency_score']=1/(1+d['years_since_recent_flood'])
    d['log_population_secondary']=np.log(d['population_secondary'].where(d['population_secondary']>0))
features=['log_event_count','log_mean_duration','log_total_duration','log_past_fatalities','log_past_injuries','log_fatal_event_count','log_distinct_flood_years','recency_score']
# Standardization learned from development transition only.
means=dev[features].mean(); stds=dev[features].std(ddof=0).replace(0,1)
for d in [dev,test]:
    for c in features:
        d[c+'_z']=(d[c]-means[c])/stds[c]
# Benchmark score formulas, with standardization for equal composites learned on dev.
def z_apply(train, target, cols, prefix):
    mu=train[cols].mean(); sd=train[cols].std(ddof=0).replace(0,1)
    for d in [dev,test]:
        vals=[]
        for c in cols:
            vals.append((d[c]-mu[c])/sd[c])
        d[prefix]=np.vstack(vals).mean(axis=0)
z_apply(dev,test,['log_event_count','log_mean_duration','log_distinct_flood_years'],'equal_weight_historical_hazard_score')
z_apply(dev,test,['log_past_fatalities','log_fatal_event_count'],'historical_persistence_score')
z_apply(dev,test,['log_event_count','log_mean_duration','log_distinct_flood_years','log_population_secondary'],'hazard_plus_population_secondary')
for d in [dev,test]:
    d['past_fatalities_score']=d['log_past_fatalities']; d['past_fatal_event_count_score']=d['log_fatal_event_count']; d['past_injury_count_score']=d['log_past_injuries']; d['past_flood_event_count_score']=d['log_event_count']; d['past_mean_duration_score']=d['log_mean_duration']; d['past_total_duration_score']=d['log_total_duration']; d['population_alone_secondary']=d['log_population_secondary']

# Regularized Poisson model with alpha fixed before final test; transformations chosen from development only.
X_cols=[c+'_z' for c in features]
X_dev=dev[X_cols].fillna(0).to_numpy(float); y_dev=dev['future_fatalities'].to_numpy(float); X_test=test[X_cols].fillna(0).to_numpy(float); y_test=test['future_fatalities'].to_numpy(float)
alpha=1.0
def poisson_obj(theta,X,y):
    intercept=theta[0]; beta=theta[1:]; eta=np.clip(intercept+X@beta,-20,20); mu=np.exp(eta); nll=np.mean(mu-y*eta)+0.5*alpha*np.sum(beta**2); grad0=np.mean(mu-y); gradb=(X.T@(mu-y))/len(y)+alpha*beta; return nll,np.r_[grad0,gradb]
init=np.zeros(X_dev.shape[1]+1); init[0]=np.log(max(y_dev.mean(),1e-6))
res=optimize.minimize(lambda th: poisson_obj(th,X_dev,y_dev), init, jac=True, method='L-BFGS-B', options={'maxiter':1000})
assert res.success, res.message
theta=res.x
def poisson_predict(X): return np.exp(np.clip(theta[0]+X@theta[1:],-20,20))
dev['primary_poisson_predicted_fatalities']=poisson_predict(X_dev); test['primary_poisson_predicted_fatalities']=poisson_predict(X_test)
# Secondary restrained nonlinear benchmark: predetermined log-target feature interaction score, no tuning.
for d in [dev,test]:
    d['secondary_log_linear_risk_score']=0.35*d['log_past_fatalities']+0.20*d['log_event_count']+0.20*d['log_total_duration']+0.15*d['log_past_injuries']+0.10*d['log_distinct_flood_years']
# Count calibration for benchmark MAE: fit one-predictor Poisson calibrator on development only.
benchmarks=[('primary_poisson_predicted_fatalities','Regularized Poisson historical-impact model','primary_model',True),('secondary_log_linear_risk_score','Restrained log-linear risk score','secondary_model',False),('past_fatalities_score','Past fatalities','benchmark',False),('past_fatal_event_count_score','Past fatal-event count','benchmark',False),('past_injury_count_score','Past injury count','benchmark',False),('past_flood_event_count_score','Past flood-event count','benchmark',False),('past_mean_duration_score','Past mean duration','benchmark',False),('past_total_duration_score','Past total duration','benchmark',False),('equal_weight_historical_hazard_score','Equal-weight historical hazard score','benchmark',False),('historical_persistence_score','Historical persistence score','benchmark',False),('population_alone_secondary','Population alone secondary','secondary_sensitivity',False),('hazard_plus_population_secondary','Hazard plus population secondary','secondary_sensitivity',False)]
calibrators={}
for score,label,group,is_count in benchmarks:
    if is_count:
        calibrators[score]=('direct',None)
    else:
        x=dev[[score]].replace([np.inf,-np.inf],np.nan).fillna(dev[score].median()).to_numpy(float); mu=x.mean(axis=0); sd=x.std(axis=0); sd[sd==0]=1; xs=(x-mu)/sd
        def obj(th): return poisson_obj(th,xs,y_dev)
        init=np.array([np.log(max(y_dev.mean(),1e-6)),0.0]); rr=optimize.minimize(lambda th: obj(th), init, jac=True, method='L-BFGS-B')
        calibrators[score]=('poisson_1d',(rr.x,mu,sd))
        for d in [dev,test]:
            xx=d[[score]].replace([np.inf,-np.inf],np.nan).fillna(dev[score].median()).to_numpy(float); z=(xx-mu)/sd; d[score+'_calibrated_predicted_fatalities']=np.exp(np.clip(rr.x[0]+z@rr.x[1:],-20,20))

def metrics_for(df, score, count_pred_col=None):
    keep_cols = [score, 'future_fatalities'] + [c for c in ['State_Name','district_name'] if c in df.columns]
    dat=df[keep_cols].replace([np.inf,-np.inf],np.nan).dropna(subset=[score,'future_fatalities']).copy()
    k=int(np.ceil(0.10*len(dat)))
    pred_top=mark_top(dat,score,k=k,ascending=False); actual_top=mark_top(dat,'future_fatalities',k=k,ascending=False)
    overlap=len(pred_top&actual_top); union=len(pred_top|actual_top)
    order=dat.sort_values(score,ascending=False).index.tolist(); gains=dat.loc[order,'future_fatalities'].to_numpy(float); ideal=np.sort(dat['future_fatalities'].to_numpy(float))[::-1]
    # Use linear fatality-count gains; exponential gains can make one disaster dominate NDCG numerically.
    dcg=np.sum(gains[:k]/np.log2(np.arange(2,k+2))); idcg=np.sum(ideal[:k]/np.log2(np.arange(2,k+2))); ndcg=dcg/idcg if idcg>0 else np.nan
    # Average precision for actual top-k binary labels.
    hits=0; prec=[]
    for rank,idx in enumerate(order, start=1):
        if idx in actual_top:
            hits+=1; prec.append(hits/rank)
    ap=float(np.mean(prec)) if prec else 0.0
    if count_pred_col and count_pred_col in df.columns:
        mae=float(np.mean(np.abs(df[count_pred_col]-df['future_fatalities'])))
    else: mae=np.nan
    return {'usable_districts':len(dat),'spearman_correlation':safe_spearman(dat[score],dat['future_fatalities']),'kendall_correlation':safe_kendall(dat[score],dat['future_fatalities']),'top10_n':k,'top10_overlap_count':overlap,'top10_recall':overlap/len(actual_top) if actual_top else np.nan,'jaccard_similarity':overlap/union if union else np.nan,'ndcg_at_top10':ndcg,'average_precision_top10':ap,'mean_absolute_error':mae}
results=[]
for score,label,group,is_count in benchmarks:
    pred_col=score if is_count else score+'_calibrated_predicted_fatalities'
    row=metrics_for(test,score,pred_col); row.update({'method':score,'method_label':label,'method_group':group,'transition':'final_test','tie_policy':'Fixed top-decile count; ties broken deterministically by state and district name.'}); results.append(row)
results=pd.DataFrame(results); results.to_csv(TABLE_DIR/'predictive_ranking_final_test_results.csv',index=False)
results[['method','method_label','method_group','top10_n','top10_overlap_count','top10_recall','jaccard_similarity','ndcg_at_top10','average_precision_top10']].to_csv(TABLE_DIR/'predictive_ranking_top10_results.csv',index=False)
# District predictions.
pred=test.copy(); pred['predicted_future_fatalities']=pred['primary_poisson_predicted_fatalities']; pred['predicted_rank']=pred.sort_values(['predicted_future_fatalities','State_Name','district_name'],ascending=[False,True,True]).reset_index().reset_index().set_index('index')['level_0'].add(1).reindex(pred.index).astype(int); pred['actual_future_fatalities']=pred['future_fatalities']; pred['predicted_top10_status']=pred.index.isin(mark_top(pred,'predicted_future_fatalities',k=TOPK,ascending=False)); pred['actual_top10_status']=pred['actual_top10']
keep=['baseline_district_id','State_Name','district_name','predicted_future_fatalities','predicted_rank','actual_future_fatalities','actual_rank','predicted_top10_status','actual_top10_status']+[b[0] for b in benchmarks if b[0] in pred.columns]
pred[keep].to_csv(TABLE_DIR/'predictive_ranking_district_predictions.csv',index=False)
# Calibration quintiles.
cal=[]
for score,label,group,is_count in benchmarks:
    pred_col=score if is_count else score+'_calibrated_predicted_fatalities'
    tmp=test[[pred_col,'future_fatalities']].copy(); tmp['risk_quintile']=pd.qcut(tmp[pred_col].rank(method='first'),5,labels=['Q1 lowest','Q2','Q3','Q4','Q5 highest'])
    for q,g in tmp.groupby('risk_quintile',observed=False): cal.append({'method':score,'method_label':label,'risk_quintile':str(q),'districts':len(g),'mean_predicted_fatalities':float(g[pred_col].mean()),'mean_actual_fatalities':float(g['future_fatalities'].mean()),'median_actual_fatalities':float(g['future_fatalities'].median()),'total_actual_fatalities':float(g['future_fatalities'].sum())})
cal=pd.DataFrame(cal); cal.to_csv(TABLE_DIR/'predictive_ranking_calibration.csv',index=False)
# Bootstrap comparisons.
compare_methods=['past_fatalities_score','past_flood_event_count_score','equal_weight_historical_hazard_score','historical_persistence_score','population_alone_secondary']
def metric_triplet(df,score):
    m=metrics_for(df,score); return m['spearman_correlation'],m['ndcg_at_top10'],m['top10_recall']
def metric_triplet_arrays(y, score):
    y=np.asarray(y,float); score=np.asarray(score,float)
    mask=np.isfinite(y) & np.isfinite(score)
    y=y[mask]; score=score[mask]
    n=len(y)
    if n<4 or len(np.unique(y))<2 or len(np.unique(score))<2:
        spearman=np.nan
    else:
        spearman=float(stats.spearmanr(score,y).statistic)
    k=int(np.ceil(0.10*n))
    pos=np.arange(n)
    pred_order=np.lexsort((pos, -score))
    actual_order=np.lexsort((pos, -y))
    pred_top=set(pred_order[:k].tolist()); actual_top=set(actual_order[:k].tolist())
    overlap=len(pred_top & actual_top)
    gains=y[pred_order]; ideal=np.sort(y)[::-1]
    dcg=np.sum(gains[:k]/np.log2(np.arange(2,k+2))); idcg=np.sum(ideal[:k]/np.log2(np.arange(2,k+2)))
    ndcg=float(dcg/idcg) if idcg>0 else np.nan
    recall=float(overlap/len(actual_top)) if actual_top else np.nan
    return spearman, ndcg, recall

def bootstrap_compare(block=False):
    rng=np.random.default_rng(RANDOM_SEED)
    base=test.reset_index(drop=True)
    y_all=base['future_fatalities'].to_numpy(float)
    primary_all=base['primary_poisson_predicted_fatalities'].to_numpy(float)
    score_arrays={comp: base[comp].replace([np.inf,-np.inf],np.nan).to_numpy(float) for comp in compare_methods}
    n=len(base)
    if block:
        state_to_idx={s: base.index[base['State_Name'].eq(s)].to_numpy() for s in sorted(base['State_Name'].unique())}
        states=np.array(list(state_to_idx.keys()))
        sample_indices=[np.concatenate([state_to_idx[s] for s in rng.choice(states,size=len(states),replace=True)]) for _ in range(BOOTSTRAPS)]
    else:
        sample_indices=[rng.integers(0,n,size=n) for _ in range(BOOTSTRAPS)]
    primary_metrics=[metric_triplet_arrays(y_all[idx], primary_all[idx]) for idx in sample_indices]
    rows=[]; metric_names=['spearman','ndcg_at_top10','top10_recall']
    obs_primary=metric_triplet(test,'primary_poisson_predicted_fatalities')
    for comp in compare_methods:
        obs_comp=metric_triplet(test,comp)
        comp_metrics=[metric_triplet_arrays(y_all[idx], score_arrays[comp][idx]) for idx in sample_indices]
        for j,metric in enumerate(metric_names):
            vals=[]
            for a,b in zip(primary_metrics, comp_metrics):
                av,bv=a[j],b[j]
                if np.isfinite(av) and np.isfinite(bv):
                    vals.append(av-bv)
            vals=np.asarray(vals,float)
            obs_a,obs_b=obs_primary[j],obs_comp[j]
            if len(vals)==0:
                med=ci_low=ci_high=prop=np.nan
            else:
                med=float(np.nanmedian(vals)); ci_low=float(np.nanpercentile(vals,2.5)); ci_high=float(np.nanpercentile(vals,97.5)); prop=float(np.nanmean(vals>0))
            rows.append({'comparison':'primary_model_minus_'+comp,'comparison_method':comp,'metric':metric,'bootstrap_type':'state_block' if block else 'ordinary_district','observed_difference':obs_a-obs_b,'bootstrap_median_difference':med,'bootstrap_ci_low':ci_low,'bootstrap_ci_high':ci_high,'proportion_primary_better':prop,'bootstrap_samples':len(vals)})
    return pd.DataFrame(rows)
ordinary=bootstrap_compare(False); stateblock=bootstrap_compare(True); ordinary.to_csv(TABLE_DIR/'predictive_ranking_paired_bootstrap.csv',index=False); stateblock.to_csv(TABLE_DIR/'predictive_ranking_state_block_bootstrap.csv',index=False)
# Geographic robustness.
state_rows=[]
for state,g in test.groupby('State_Name'):
    if len(g)>=10 and g['future_fatalities'].nunique()>1:
        m=metrics_for(g,'primary_poisson_predicted_fatalities'); m.update({'State_Name':state}); state_rows.append(m)
pd.DataFrame(state_rows).to_csv(TABLE_DIR/'predictive_ranking_state_results.csv',index=False)
loso=[]
for state in sorted(test['State_Name'].unique()):
    g=test[test['State_Name']!=state]
    m=metrics_for(g,'primary_poisson_predicted_fatalities'); m.update({'excluded_state':state,'remaining_districts':len(g),'excluded_future_fatalities':float(test.loc[test['State_Name']==state,'future_fatalities'].sum())}); loso.append(m)
pd.DataFrame(loso).to_csv(TABLE_DIR/'predictive_ranking_leave_one_state_out.csv',index=False)
# Weak reporting states: bottom 20% fatality reporting in 2005-2023 raw states.
weak_states=set(state_comp[state_comp['period'].eq('2005-2023')].sort_values('fatality_reporting_percent').head(max(1,int(.2*state_comp[state_comp['period'].eq('2005-2023')]['State_raw'].nunique())))['State_raw'].apply(norm_text))
test['state_norm']=test['State_Name'].apply(norm_text)
zero_hist=test[test['fatality_count']==0]; excl_weak=test[~test['state_norm'].isin(weak_states)]
infl=[]
for label,g in [('all_final',test),('zero_historical_fatalities',zero_hist),('excluding_weak_reporting_states',excl_weak)]:
    m=metrics_for(g,'primary_poisson_predicted_fatalities'); m.update({'diagnostic_subset':label,'districts':len(g),'future_fatalities_sum':float(g['future_fatalities'].sum())}); infl.append(m)
# Dominance diagnostics.
long=pd.read_csv(TABLE_DIR/'predictive_ranking_event_district_long.csv'); outcome_events=long[(long['year'].between(2005,2023))&(long['district_specific_impact'].astype(str).str.lower().isin(['true','1']))].copy(); top_events=outcome_events.sort_values('fatalities_for_district_primary',ascending=False).head(10)
infl.append({'diagnostic_subset':'top_single_district_event_share','districts':len(test),'future_fatalities_sum':float(test['future_fatalities'].sum()),'spearman_correlation':safe_spearman(test['primary_poisson_predicted_fatalities'],test['future_fatalities']),'top_event_fatalities':float(top_events['fatalities_for_district_primary'].iloc[0]) if len(top_events) else np.nan,'top_two_event_share':float(top_events['fatalities_for_district_primary'].head(2).sum()/test['future_fatalities'].sum()) if test['future_fatalities'].sum()>0 else np.nan,'top_ten_event_share':float(top_events['fatalities_for_district_primary'].head(10).sum()/test['future_fatalities'].sum()) if test['future_fatalities'].sum()>0 else np.nan})
pd.DataFrame(infl).to_csv(TABLE_DIR/'predictive_ranking_influence_diagnostics.csv',index=False)
# Figures.
fig,ax=plt.subplots(figsize=(6.5,5),constrained_layout=True); ax.scatter(test['primary_poisson_predicted_fatalities'],test['future_fatalities'],s=20,alpha=.7); ax.set_xlabel('Predicted future fatalities'); ax.set_ylabel('Actual future fatalities'); ax.set_title('Final temporal test: actual vs predicted fatalities'); ax.text(.02,.95,f'n={len(test)}\nSpearman={safe_spearman(test["primary_poisson_predicted_fatalities"],test["future_fatalities"]):.3f}',transform=ax.transAxes,va='top'); fig.savefig(FIGURE_DIR/'predictive_ranking_actual_vs_predicted.png',dpi=220,bbox_inches='tight'); plt.close(fig)
fig,ax=plt.subplots(figsize=(9,5.5),constrained_layout=True); plot=results.sort_values('spearman_correlation'); ax.barh(plot['method_label'],plot['spearman_correlation'],color='#4c78a8'); ax.set_xlabel('Spearman with future fatalities'); ax.set_title('Final test model and benchmark ranking performance'); fig.savefig(FIGURE_DIR/'predictive_ranking_model_comparison.png',dpi=220,bbox_inches='tight'); plt.close(fig)
fig,ax=plt.subplots(figsize=(8,5),constrained_layout=True); topplot=results.sort_values('top10_recall'); ax.barh(topplot['method_label'],topplot['top10_recall'],color='#5a8f5a'); ax.set_xlabel('Recall of actual future-fatality top 10%'); ax.set_title('Top-10% agreement in final temporal test'); fig.savefig(FIGURE_DIR/'predictive_ranking_top10_agreement.png',dpi=220,bbox_inches='tight'); plt.close(fig)
fig,ax=plt.subplots(figsize=(7,5),constrained_layout=True); c=cal[cal['method'].eq('primary_poisson_predicted_fatalities')]; ax.plot(c['risk_quintile'],c['mean_predicted_fatalities'],marker='o',label='Predicted'); ax.plot(c['risk_quintile'],c['mean_actual_fatalities'],marker='o',label='Actual'); ax.set_ylabel('Mean fatalities'); ax.set_title('Calibration by predicted-risk quintile'); ax.legend(); fig.savefig(FIGURE_DIR/'predictive_ranking_calibration.png',dpi=220,bbox_inches='tight'); plt.close(fig)
# Reporting figure already produced in 13a; assert exists.
assert (FIGURE_DIR/'predictive_ranking_reporting_completeness.png').exists()
# Final decision.
primary=results.set_index('method').loc['primary_poisson_predicted_fatalities']; past=results.set_index('method').loc['past_fatalities_score']; event=results.set_index('method').loc['past_flood_event_count_score']; pers=results.set_index('method').loc['historical_persistence_score']
state_comp_ok=stateblock[(stateblock['comparison_method'].isin(['past_fatalities_score','past_flood_event_count_score','equal_weight_historical_hazard_score','historical_persistence_score'])) & (stateblock['metric'].eq('spearman'))]
clear_outperform=(primary['spearman_correlation']>max(past['spearman_correlation'],event['spearman_correlation'],pers['spearman_correlation'])+0.03) and (state_comp_ok['bootstrap_ci_low']>0).any()
top_improves=primary['top10_recall']>=max(past['top10_recall'],event['top10_recall'],pers['top10_recall'])+0.05
not_dominated=pd.read_csv(TABLE_DIR/'predictive_ranking_influence_diagnostics.csv').query("diagnostic_subset=='top_single_district_event_share'")['top_two_event_share'].iloc[0] < 0.35
if clear_outperform and top_improves and not_dominated:
    decision='GO — predictive-ranking paper'
elif primary['spearman_correlation']>0.15 and primary['top10_recall']>0.15:
    decision='GO WITH A NARROW CLAIM'
else:
    decision='NO-GO'
report=f"""# Predictive Ranking Final Go/No-Go Report

## Decision

**{decision}**

## Final Ranking Meaning

Districts are ranked by predicted recorded flood-fatality burden in 2005-2023 using only district flood-history information from 1986-2004. Full-period DFSI, full-period flooded area, full-period GFD metrics and outcome-period information were not used as predictors.

## Data and Feasibility

- Event-level input: `data/raw/India_Flood_Inventory_v3.csv`.
- Final test districts: {len(test)}.
- Future-fatality top-decile count: {TOPK}.
- Final-period total primary assignable fatalities: {test['future_fatalities'].sum():.0f}.
- Districts with zero future fatalities: {(test['future_fatalities']==0).sum()} ({(test['future_fatalities']==0).mean()*100:.1f}%).
- Human-impact counts are assigned only from district-specific or single-district event records; multi-district event death totals are audited but not duplicated across districts.

## Final Test Performance

- Primary Poisson model Spearman: {primary['spearman_correlation']:.4f}; Kendall: {primary['kendall_correlation']:.4f}; top-10 recall: {primary['top10_recall']:.4f}; NDCG@top10: {primary['ndcg_at_top10']:.4f}; average precision: {primary['average_precision_top10']:.4f}; MAE: {primary['mean_absolute_error']:.4f}.
- Past fatalities Spearman: {past['spearman_correlation']:.4f}; top-10 recall: {past['top10_recall']:.4f}.
- Past flood-event count Spearman: {event['spearman_correlation']:.4f}; top-10 recall: {event['top10_recall']:.4f}.
- Historical persistence score Spearman: {pers['spearman_correlation']:.4f}; top-10 recall: {pers['top10_recall']:.4f}.

## Interpretation

This is a strict temporal prediction test, not a retrospective composite-index association. The result supports a narrow predictive claim only if the primary model improves decision-relevant ranking over simple historical baselines and survives state-block uncertainty.

## Limitations

- Fatality reporting is incomplete and changes across periods.
- Multi-district fatality totals are not allocated to individual districts, so district fatality burden is conservative and may undercount large regional disasters.
- Many districts have zero recorded future fatalities, creating ties and sparse outcomes.
- Population is only a secondary sensitivity predictor because its reference year is not documented.
- District boundary changes and unmatched/ambiguous event records remain unresolved.
"""
(REPORT_DIR/'predictive_ranking_final_go_no_go.md').write_text(report,encoding='utf-8')
closure=f"""# Quantitative Analysis Closure

## Questions Tested

1. DFSI reproducibility and ranking agreement.
2. DFSI sensitivity to fatality weights, component removal, Monte Carlo weights and priority cutoffs.
3. Convergent validity against Global Flood Database satellite inundation.
4. Separation of physical hazard, exposure and human consequences.
5. Leakage-free temporal prediction of future recorded flood-fatality burden.

## Satellite Validation

Satellite-observed physical inundation corresponded more strongly with physical-hazard-only measures than with the full six-component DFSI, supporting construct separation.

## Weight Sensitivity

Weight changes left national rank correlations high but altered district priority membership, showing that rank correlation alone is insufficient for priority robustness.

## Future-Impact Ranking

Decision: {decision}. The final ranking target is recorded flood fatalities in 2005-2023 predicted from 1986-2004 district history only.

## Defensible Claims

- Physical inundation should be evaluated separately from exposure and recorded human consequences.
- Historical flood records can be tested prospectively only with strict non-overlapping windows.
- Reporting drift and sparse fatality outcomes limit strong predictive claims.

## Not Defensible

- Full-period DFSI should not be described as predicting outcomes inside the same period.
- Full-period flooded-area or GFD summaries should not be used as temporal predictors.
- Multi-district event fatalities should not be duplicated across districts.

## Recommended Paper Question

Can historical district flood records identify districts at elevated future recorded flood-fatality burden, and how does this compare with separating physical hazard from exposure and consequences?

## Manuscript Recommendation

Rebuild or narrow the manuscript according to `outputs/reports/predictive_ranking_final_go_no_go.md`; do not retain claims that require leaked full-period predictors.

## Retain for Main Paper

- Predictive temporal feasibility and final ranking tables.
- Final model/benchmark comparison.
- Satellite physical-hazard validation as construct-validity support.
- Reporting completeness figure.

## Supplementary or Exclude

- Detailed DFSI arithmetic debugging should remain out of the manuscript.
- Monte Carlo DFSI sensitivity can be supplementary unless central to framing.
- In-sample weight optimisation should be excluded from causal or predictive claims.

## Frozen Final Analysis Files

- `notebooks/13a_temporal_prediction_feasibility.ipynb`
- `notebooks/13b_district_period_panel.ipynb`
- `notebooks/13c_future_impact_ranking.ipynb`
- `outputs/reports/predictive_ranking_final_go_no_go.md`
- `outputs/reports/quantitative_analysis_closure.md`

## Unresolved Data Requirements

- District-specific allocation of deaths and injuries for multi-district events.
- Administrative-boundary crosswalks over time.
- Population reference year/source.
- Reporting completeness metadata sufficient to distinguish missing from true zero impacts.
"""
(REPORT_DIR/'quantitative_analysis_closure.md').write_text(closure,encoding='utf-8')
for path in [TABLE_DIR/'predictive_ranking_final_test_results.csv',TABLE_DIR/'predictive_ranking_top10_results.csv',TABLE_DIR/'predictive_ranking_calibration.csv',TABLE_DIR/'predictive_ranking_district_predictions.csv',TABLE_DIR/'predictive_ranking_paired_bootstrap.csv',TABLE_DIR/'predictive_ranking_state_block_bootstrap.csv',TABLE_DIR/'predictive_ranking_state_results.csv',TABLE_DIR/'predictive_ranking_leave_one_state_out.csv',TABLE_DIR/'predictive_ranking_influence_diagnostics.csv',FIGURE_DIR/'predictive_ranking_actual_vs_predicted.png',FIGURE_DIR/'predictive_ranking_model_comparison.png',FIGURE_DIR/'predictive_ranking_top10_agreement.png',FIGURE_DIR/'predictive_ranking_calibration.png',FIGURE_DIR/'predictive_ranking_reporting_completeness.png',REPORT_DIR/'predictive_ranking_final_go_no_go.md',REPORT_DIR/'quantitative_analysis_closure.md']:
    assert path.exists(), path
print(report)
display(results[['method_label','spearman_correlation','kendall_correlation','top10_recall','jaccard_similarity','ndcg_at_top10','average_precision_top10','mean_absolute_error']])
display(ordinary.head(15)); display(stateblock.head(15)); display(pd.DataFrame(outdist))
RAW_HASHES_AFTER={p.name: sha256_file(p) for p in sorted(RAW_DIR.glob('*.csv'))}
assert RAW_HASHES_BEFORE==RAW_HASHES_AFTER, 'data/raw changed during predictive ranking.'


# Predictive Ranking Final Go/No-Go Report

## Decision

**GO WITH A NARROW CLAIM**

## Final Ranking Meaning

Districts are ranked by predicted recorded flood-fatality burden in 2005-2023 using only district flood-history information from 1986-2004. Full-period DFSI, full-period flooded area, full-period GFD metrics and outcome-period information were not used as predictors.

## Data and Feasibility

- Event-level input: `data/raw/India_Flood_Inventory_v3.csv`.
- Final test districts: 720.
- Future-fatality top-decile count: 72.
- Final-period total primary assignable fatalities: 3654.
- Districts with zero future fatalities: 388 (53.9%).
- Human-impact counts are assigned only from district-specific or single-district event records; multi-district event death totals are audited but not duplicated across districts.

## Final Test Performance

- Primary Poisson model Spearman: 0.4614; Kendall: 0.3607; top-10 recall: 0.4306; NDCG@top10: 0.6884; average precision: 0.4032; MAE: 6.2942.
- 

,method_label,spearman_correlation,kendall_correlation,top10_recall,jaccard_similarity,ndcg_at_top10,average_precision_top10,mean_absolute_error
0,Regularized Poisson historical-impact model,0.461418,0.360657,0.430556,0.274336,0.688419,0.403156,6.294199
1,Restrained log-linear risk score,0.496805,0.389744,0.402778,0.252174,0.696654,0.421007,5.810417
2,Past fatalities,0.359534,0.311373,0.375000,0.230769,0.446070,0.339556,6.172537
3,Past fatal-event count,0.374836,0.329449,0.458333,0.297297,0.679062,0.384216,6.559359
4,Past injury count,0.343054,0.304859,0.430556,0.274336,0.572042,0.328515,6.228320
5,Past flood-event count,0.492953,0.392108,0.388889,0.241379,0.685980,0.385103,6.229096
6,Past mean duration,0.212492,0.154136,0.055556,0.028571,0.056156,0.092510,6.222550
7,Past total duration,0.413690,0.321126,0.097222,0.051095,0.138572,0.185108,6.155683
8,Equal-weight historical hazard score,0.462626,0.361294,0.263889,0.152000,0.311012,0.240105,6.179888
9,Historical persistence score,0.368250,0.319172,0.416667,0.263158,0.675578,0.390157,6.172257


,comparison,comparison_method,metric,bootstrap_type,observed_difference,bootstrap_median_difference,bootstrap_ci_low,bootstrap_ci_high,proportion_primary_better,bootstrap_samples
0,primary_model_minus_past_fatalities_score,past_fatalities_score,spearman,ordinary_district,0.101884,0.101565,0.041010,0.162940,0.9990,2000
1,primary_model_minus_past_fatalities_score,past_fatalities_score,ndcg_at_top10,ordinary_district,0.242349,0.222167,-0.046049,0.346508,0.9245,2000
2,primary_model_minus_past_fatalities_score,past_fatalities_score,top10_recall,ordinary_district,0.055556,0.048611,-0.027778,0.138889,0.8470,2000
3,primary_model_minus_past_flood_event_count_score,past_flood_event_count_score,spearman,ordinary_district,-0.031535,-0.031159,-0.063730,0.000812,0.0295,2000
4,primary_model_minus_past_flood_event_count_score,past_flood_event_count_score,ndcg_at_top10,ordinary_district,0.002439,0.004072,-0.114828,0.133236,0.5295,2000
5,primary_model_minus_past_flood_event_count_score,past_flood_event_count_score,top10_recall,ordinary_district,0.041667,0.000000,-0.125000,0.125000,0.4560,2000
6,primary_model_minus_equal_weight_historical_ha...,equal_weight_historical_hazard_score,spearman,ordinary_district,-0.001208,-0.000583,-0.037438,0.034286,0.4775,2000
7,primary_model_minus_equal_weight_historical_ha...,equal_weight_historical_hazard_score,ndcg_at_top10,ordinary_district,0.377407,0.366424,0.070675,0.501516,0.9980,2000
8,primary_model_minus_equal_weight_historical_ha...,equal_weight_historical_hazard_score,top10_recall,ordinary_district,0.166667,0.138889,0.027431,0.263889,0.9860,2000
9,primary_model_minus_historical_persistence_score,historical_persistence_score,spearman,ordinary_district,0.093168,0.092758,0.032033,0.154480,0.9980,2000


,comparison,comparison_method,metric,bootstrap_type,observed_difference,bootstrap_median_difference,bootstrap_ci_low,bootstrap_ci_high,proportion_primary_better,bootstrap_samples
0,primary_model_minus_past_fatalities_score,past_fatalities_score,spearman,state_block,0.101884,0.110170,-0.021648,0.231691,0.9485,2000
1,primary_model_minus_past_fatalities_score,past_fatalities_score,ndcg_at_top10,state_block,0.242349,0.218878,-0.086283,0.347623,0.8740,2000
2,primary_model_minus_past_fatalities_score,past_fatalities_score,top10_recall,state_block,0.055556,0.051724,-0.024691,0.134349,0.8920,2000
3,primary_model_minus_past_flood_event_count_score,past_flood_event_count_score,spearman,state_block,-0.031535,-0.029206,-0.095089,0.030141,0.1765,2000
4,primary_model_minus_past_flood_event_count_score,past_flood_event_count_score,ndcg_at_top10,state_block,0.002439,0.006890,-0.127534,0.202058,0.5505,2000
5,primary_model_minus_past_flood_event_count_score,past_flood_event_count_score,top10_recall,state_block,0.041667,-0.012987,-0.157895,0.156639,0.4170,2000
6,primary_model_minus_equal_weight_historical_ha...,equal_weight_historical_hazard_score,spearman,state_block,-0.001208,-0.001177,-0.080939,0.086493,0.4840,2000
7,primary_model_minus_equal_weight_historical_ha...,equal_weight_historical_hazard_score,ndcg_at_top10,state_block,0.377407,0.327655,-0.034439,0.557111,0.9580,2000
8,primary_model_minus_equal_weight_historical_ha...,equal_weight_historical_hazard_score,top10_recall,state_block,0.166667,0.128205,-0.071454,0.324406,0.8790,2000
9,primary_model_minus_historical_persistence_score,historical_persistence_score,spearman,state_block,0.093168,0.100598,-0.030835,0.223001,0.9360,2000


,transition,districts,total_future_fatalities,positive_future_fatality_districts,zero_future_fatality_districts,zero_future_fatality_percent,max_future_fatalities,top10_count
0,development,720,3202.0,162,558,77.500000,263.0,72
1,final_test,720,3654.0,332,388,53.888889,338.0,72
